# The Billion-Dollar Club: Reproducible Unicorn Analysis

**Study window: 2019–2021.** This notebook complements the PostgreSQL solution and historical dashboard. Data are not current; valuations are snapshot fields, not historic valuations at the join date.

In [1]:
from pathlib import Path
import pandas as pd, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
s = json.loads((ROOT / 'data/verified_source_snapshot.json').read_text())
print('Full source size:',s['full_dataset_company_count'])
print('2019–2021 arrivals:',s['scope_count'])
print('Three-industry share: {:.1%}'.format(s['top3_scope_count']/s['scope_count']))

Full source size: 1074
2019–2021 arrivals: 732
Three-industry share: 54.6%


## 01 · Original SQL answer, independently verified
The strongest three industries are selected using the **combined 2019–2021 count**, not their valuation or rank within an individual year.

In [2]:
answer = pd.read_csv(ROOT / 'results/01_top3_industry_2019_2021.csv')
answer

                                industry  year  num_unicorns  average_valuation_billions
0                                Fintech  2021           138                        2.75
1           Internet software & services  2021           119                        2.15
2        E-commerce & direct-to-consumer  2021            47                        2.47
3           Internet software & services  2020            20                        4.35
4        E-commerce & direct-to-consumer  2020            16                        4.00
5                                Fintech  2020            15                        4.33
6                                Fintech  2019            20                        6.80
7           Internet software & services  2019            13                        4.23
8        E-commerce & direct-to-consumer  2019            12                        2.58

## 02 · Run the full source-level analysis
First execute `python scripts/download_source.py` from the repository root with internet access. These raw CSVs are intentionally **not** redistributed in the ZIP. The local script performs one-to-one joins and revalidates the cohort output against the snapshot above.

In [ ]:
names = ('companies','dates','funding','industries')
ready = all((ROOT/'data'/'raw'/f'{name}.csv').exists() for name in names)
print('Raw CSVs available:', ready)
if ready:
    tables = {name: pd.read_csv(ROOT/'data'/'raw'/f'{name}.csv') for name in names}
    print({k: len(v) for k,v in tables.items()})
else:
    print('First run: python scripts/download_source.py')

## 03 · Analytical checks and limitations
- Do not use valuation as the ranking key: the assignment ranks *new company counts*.
- Parse `date_joined` with `dayfirst=True`.
- Strip extra literal quotes in some mirror industry labels.
- Validate `company_id` uniqueness in every source table before merging.
- Keep historical snapshot valuations distinct from year-joined cohort labels.
- The mirror has 16 missing cities, 13 zero-funding entries and one impossible founding/year-joined ordering.

In [3]:
years = pd.read_csv(ROOT/'results/02_yearly_growth.csv')
assert years.all_new_unicorns.sum() == 732
assert years.top3_new_unicorns.sum() == 400
years

   year  all_new_unicorns  top3_new_unicorns  top3_avg_valuation_billions
0  2019               104                 45                         4.93
1  2020               108                 51                         4.24
2  2021               520                304                         2.47

## 04 · Visual interpretation
The charts below are generated from the verified source-query outputs. Each bar compares **counts**, never a sum of averages. For a reproducible rebuild, run `python scripts/plot_results.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
source = pd.read_csv(ROOT/'results/01_top3_industry_2019_2021.csv')
plt.figure(figsize=(9,4))
for year in (2019,2020,2021):
    x = source[source.year.eq(year)].set_index('industry').num_unicorns
    print(year, x.to_dict())

![Annual cohorts by sector](../charts/industry_year_comparison.png)

![Geographic concentration](../charts/country_distribution.png)